# 02 — ColMaps Feature Scope Definition

This notebook defines the geographic feature scope required by ColMaps before
constructing the filtered OpenStreetMap dataset.

ColMaps is designed as a route-oriented tourism and exploration application.
Given an origin and destination, the application calculates a route and
subsequently recommends potentially interesting places located near that route.

The recommendations are influenced by the interests selected by the user before
the search. Instead of exposing OpenStreetMap tagging categories directly,
ColMaps defines its own user-oriented feature categories and maps them to
explicit OSM `key=value` combinations.

The objective of this notebook is therefore to establish the semantic boundary
of the ColMaps dataset before any filtering operation is performed.

No OSM objects are filtered or transformed in this notebook.

## 1. Functional Scope

The main ColMaps workflow can be summarized as:

1. The user selects an origin and destination.
2. The user selects the types of places they are interested in discovering.
3. ColMaps calculates the route between both locations.
4. Relevant geographic features located near the route are identified.
5. Candidate features are ranked or filtered according to route proximity,
   detour cost, and user preferences.

The current preprocessing stage is concerned only with step 2 from a data
modelling perspective: defining which geographic feature types should be
available as potential recommendations.

Route proximity analysis, detour optimization, recommendation ranking, and
other spatial optimization techniques are outside the scope of this notebook
and will be implemented in later stages.

## 2. ColMaps Feature Categories

OpenStreetMap organizes geographic information using a flexible tagging model.
However, OSM tag families do not correspond directly to the categories that are
useful to a traveler.

For example, the `amenity` family contains restaurants and cafés, but also
schools, waste baskets, hospitals, parking infrastructure, and many other
features outside the primary tourism scope of ColMaps.

For this reason, ColMaps introduces its own application-level feature
categories. Each category represents a user interest and can later be mapped
to one or more explicit OSM `key=value` combinations.

In [1]:
FEATURE_CATEGORIES = {
    "attractions": {
        "label": "Attractions",
        "description": "General tourist attractions and visitor-oriented places.",
    },
    "museums_culture": {
        "label": "Museums & Culture",
        "description": "Museums, galleries, artworks, theatres, and cultural places.",
    },
    "history_heritage": {
        "label": "History & Heritage",
        "description": "Monuments, memorials, archaeological sites, ruins, and historic landmarks.",
    },
    "viewpoints": {
        "label": "Viewpoints",
        "description": "Places intended for scenic or panoramic observation.",
    },
    "nature": {
        "label": "Nature",
        "description": "Natural geographic features potentially relevant to travelers.",
    },
    "parks_recreation": {
        "label": "Parks & Recreation",
        "description": "Parks, gardens, recreational spaces, leisure facilities, and activities.",
    },
    "food_drink": {
        "label": "Food & Drink",
        "description": "Restaurants, cafés, bars, pubs, and similar visitor services.",
    },
    "accommodation": {
        "label": "Accommodation",
        "description": "Hotels, hostels, guest houses, campsites, and other lodging.",
    },
    "entertainment": {
        "label": "Entertainment",
        "description": "Zoos, theme parks, aquariums, water parks, and other entertainment destinations.",
    },
    "markets_shopping": {
        "label": "Markets & Shopping",
        "description": "Markets, shops, and commercial places potentially useful or interesting to travelers.",
    },
    "health_pharmacy": {
        "label": "Health & Pharmacy",
        "description": "Pharmacies, hospitals, clinics, and other healthcare services potentially useful during a journey.",
    },
    "transport_travel": {
        "label": "Transport & Travel",
        "description": "Fuel, parking, transport facilities, and other services useful during a journey.",
    },
}

In [2]:
import pandas as pd

feature_categories = pd.DataFrame(
    [
        {
            "category": key,
            "label": value["label"],
            "description": value["description"],
        }
        for key, value in FEATURE_CATEGORIES.items()
    ]
)

feature_categories

,category,label,description
0,attractions,Attractions,General tourist attractions and visitor-orient...
1,museums_culture,Museums & Culture,"Museums, galleries, artworks, theatres, and cu..."
2,history_heritage,History & Heritage,"Monuments, memorials, archaeological sites, ru..."
3,viewpoints,Viewpoints,Places intended for scenic or panoramic observ...
4,nature,Nature,Natural geographic features potentially releva...
5,parks_recreation,Parks & Recreation,"Parks, gardens, recreational spaces, leisure f..."
6,food_drink,Food & Drink,"Restaurants, cafés, bars, pubs, and similar vi..."
7,accommodation,Accommodation,"Hotels, hostels, guest houses, campsites, and ..."
8,entertainment,Entertainment,"Zoos, theme parks, aquariums, water parks, and..."
9,markets_shopping,Markets & Shopping,"Markets, shops, and commercial places potentia..."


## 3. OSM Mapping
The ColMaps feature categories defined above must be mapped to explicit
OpenStreetMap `key=value` combinations.

The mapping is based on the values observed in the Colombia GeoFabrik extract
during the raw dataset inspection performed in
`01_inspect_osm.ipynb`.

Each candidate value is evaluated according to its semantic meaning and its
relevance to the ColMaps route-oriented tourism use case.

The objective is not to retain complete OSM tag families, but to construct an
explicit and controlled mapping between ColMaps categories and selected OSM
features.

### 3.1 Attractions

The `Attractions` category represents places whose primary purpose is to be
visited or experienced by travelers.

The raw dataset inspection identified `tourism=attraction` as an explicit OSM
classification for general tourist attractions with 1191 occurences in the dataset.

This value is therefore considered a direct match for the ColMaps
`Attractions` category and man_made too.

The following categories are included:
- `tourism=attraction`
- `man_made=lighthouse`
- `man_made=observatory`

In [3]:
## General Structure to store OSM keys
FEATURE_MAPPING = []

In [4]:
FEATURE_MAPPING.extend([
    {
        "category": "attractions",
        "osm_key": "tourism",
        "osm_value": "attraction",
        "include": True,
        "reason": "Explicit OSM classification for tourist attractions.",
    },
    {
        "category": "attractions",
        "osm_key": "man_made",
        "osm_value": "lighthouse",
        "include": True,
        "reason": "Lighthouses represent distinctive landmarks with potential visitor and sightseeing relevance.",
    },
    {
        "category": "attractions",
        "osm_key": "man_made",
        "osm_value": "observatory",
        "include": True,
        "reason": "Observatories represent distinctive visitor-oriented or scientific destinations potentially relevant to travelers.",
    },
])

feature_mapping = pd.DataFrame(FEATURE_MAPPING)
feature_mapping

,category,osm_key,osm_value,include,reason
0,attractions,tourism,attraction,True,Explicit OSM classification for tourist attrac...
1,attractions,man_made,lighthouse,True,Lighthouses represent distinctive landmarks wi...
2,attractions,man_made,observatory,True,Observatories represent distinctive visitor-or...


### 3.2 Viewpoints

The `Viewpoints` category represents locations intended for observing scenic,
panoramic, or otherwise notable surroundings.

The raw dataset inspection identified `tourism=viewpoint` as a direct OSM
classification for this type of feature, with 1,040 occurrences in the
Colombia extract.

Because viewpoints directly support the route-oriented exploration purpose of
ColMaps, this value is mapped to a dedicated user-interest category rather
than being grouped under the broader `Attractions` category.

In [5]:
FEATURE_MAPPING.append({
    "category": "viewpoints",
    "osm_key": "tourism",
    "osm_value": "viewpoint",
    "include": True,
    "reason": "Explicit OSM classification for scenic viewpoints.",
})

feature_mapping = pd.DataFrame(FEATURE_MAPPING)
feature_mapping

,category,osm_key,osm_value,include,reason
0,attractions,tourism,attraction,True,Explicit OSM classification for tourist attrac...
1,attractions,man_made,lighthouse,True,Lighthouses represent distinctive landmarks wi...
2,attractions,man_made,observatory,True,Observatories represent distinctive visitor-or...
3,viewpoints,tourism,viewpoint,True,Explicit OSM classification for scenic viewpoi...


### 3.3 Museums & Culture

The `Museums & Culture` category represents places and features associated with
museums, visual arts, exhibitions, performing arts, and other cultural
experiences that may be relevant to travelers.

Unlike the previous categories, this ColMaps category does not correspond to a
single OSM `key=value` combination. The raw dataset inspection identified
several classifications that describe different types of cultural features.

The following observed values are considered candidates for this category:

- `tourism=museum`
- `tourism=gallery`
- `tourism=artwork`
- `amenity=theatre`

These values are grouped under a common ColMaps category because they represent
different forms of cultural discovery from the perspective of the traveler.

The number of matches found in phase 01 was the following:

- `tourism=museum      389`
- `tourism=gallery      45`
- `tourism=artwork   1,021`
- `amenity=theatre     393`

In [6]:
FEATURE_MAPPING.extend([
    {
        "category": "museums_culture",
        "osm_key": "tourism",
        "osm_value": "museum",
        "include": True,
        "reason": "Museums represent explicit visitor-oriented cultural destinations.",
    },
    {
        "category": "museums_culture",
        "osm_key": "tourism",
        "osm_value": "gallery",
        "include": True,
        "reason": "Art galleries represent cultural places potentially relevant to travelers.",
    },
    {
        "category": "museums_culture",
        "osm_key": "tourism",
        "osm_value": "artwork",
        "include": True,
        "reason": "Public artworks can represent cultural points of interest along a route.",
    },
    {
        "category": "museums_culture",
        "osm_key": "amenity",
        "osm_value": "theatre",
        "include": True,
        "reason": "Theatres represent cultural and performing arts destinations.",
    },
])

feature_mapping = pd.DataFrame(FEATURE_MAPPING)
feature_mapping

,category,osm_key,osm_value,include,reason
0,attractions,tourism,attraction,True,Explicit OSM classification for tourist attrac...
1,attractions,man_made,lighthouse,True,Lighthouses represent distinctive landmarks wi...
2,attractions,man_made,observatory,True,Observatories represent distinctive visitor-or...
3,viewpoints,tourism,viewpoint,True,Explicit OSM classification for scenic viewpoi...
4,museums_culture,tourism,museum,True,Museums represent explicit visitor-oriented cu...
5,museums_culture,tourism,gallery,True,Art galleries represent cultural places potent...
6,museums_culture,tourism,artwork,True,Public artworks can represent cultural points ...
7,museums_culture,amenity,theatre,True,Theatres represent cultural and performing art...


### 3.4 History & Heritage

The `History & Heritage` category represents historical places, structures,
remains, and commemorative features that may provide cultural or historical
interest during route exploration.

The raw dataset inspection identified 55 distinct values under the `historic`
key. Unlike `tourism=attraction` or `tourism=viewpoint`, the complete
`historic=*` family is not retained automatically. Individual values are
evaluated according to their potential relevance as visitor-oriented historical
features.

The selected values include major heritage features such as `monument`,
`memorial`, `archaeological_site`, `ruins`, and `castle`, as well as smaller or
more context-dependent features such as wayside_shrine, wayside_cross,
`church`, `cannon`, `wreck`, and `manor`.

Their inclusion in the feature scope does not imply that every corresponding
OSM object will become a `ColMaps` recommendation. The subsequent validation
stage will inspect the selected objects in greater detail, including their
attributes, geometry, naming quality, and practical relevance to route
exploration.

The following observed values are initially selected:

- `historic=monument`
- `historic=memorial`
- `historic=archaeological_site`
- `historic=ruins`
- `historic=castle`
- `historic=city_gate`
- `historic=citywalls`
- `historic=battlefield`
- `historic=wayside_shrine`
- `historic=wayside_cross`
- `historic=church`
- `historic=cannon`
- `historic=wreck`
- `historic=manor`
- `man_made=obelisk`
- `man_made=windmill`
- `man_made=ceremonial_gate`

In [7]:
FEATURE_MAPPING.extend([
    {
        "category": "history_heritage",
        "osm_key": "historic",
        "osm_value": "monument",
        "include": True,
        "reason": "Monuments represent explicit historical or commemorative landmarks.",
    },
    {
        "category": "history_heritage",
        "osm_key": "historic",
        "osm_value": "memorial",
        "include": True,
        "reason": "Memorials represent commemorative places of potential historical interest.",
    },
    {
        "category": "history_heritage",
        "osm_key": "historic",
        "osm_value": "archaeological_site",
        "include": True,
        "reason": "Archaeological sites represent explicit heritage locations.",
    },
    {
        "category": "history_heritage",
        "osm_key": "historic",
        "osm_value": "ruins",
        "include": True,
        "reason": "Historic ruins can represent visitor-relevant heritage features.",
    },
    {
        "category": "history_heritage",
        "osm_key": "historic",
        "osm_value": "castle",
        "include": True,
        "reason": "Historic castles represent potentially significant heritage landmarks.",
    },
    {
        "category": "history_heritage",
        "osm_key": "historic",
        "osm_value": "city_gate",
        "include": True,
        "reason": "Historic city gates can represent identifiable heritage landmarks.",
    },
    {
        "category": "history_heritage",
        "osm_key": "historic",
        "osm_value": "citywalls",
        "include": True,
        "reason": "Historic city walls can represent visitor-relevant heritage structures.",
    },
    {
        "category": "history_heritage",
        "osm_key": "historic",
        "osm_value": "battlefield",
        "include": True,
        "reason": "Historic battlefields may represent places of historical and cultural interest.",
    },
    {
        "category": "history_heritage",
        "osm_key": "historic",
        "osm_value": "wayside_shrine",
        "include": True,
        "reason": "Wayside shrines may represent small historical, religious, or cultural landmarks.",
    },
    {
        "category": "history_heritage",
        "osm_key": "historic",
        "osm_value": "wayside_cross",
        "include": True,
        "reason": "Wayside crosses may represent small historical, religious, or cultural landmarks.",
    },
    {
        "category": "history_heritage",
        "osm_key": "historic",
        "osm_value": "church",
        "include": True,
        "reason": "Historic churches may represent culturally significant heritage destinations.",
    },
    {
        "category": "history_heritage",
        "osm_key": "historic",
        "osm_value": "cannon",
        "include": True,
        "reason": "Historic cannons may represent military heritage features of visitor interest.",
    },
    {
        "category": "history_heritage",
        "osm_key": "historic",
        "osm_value": "wreck",
        "include": True,
        "reason": "Historic wrecks may represent distinctive historical or archaeological features.",
    },
    {
        "category": "history_heritage",
        "osm_key": "historic",
        "osm_value": "manor",
        "include": True,
        "reason": "Historic manors may represent architecturally and culturally significant heritage places.",
    },
        {
        "category": "history_heritage",
        "osm_key": "man_made",
        "osm_value": "obelisk",
        "include": True,
        "reason": "Obelisks may represent commemorative or historically significant landmarks.",
    },
    {
        "category": "history_heritage",
        "osm_key": "man_made",
        "osm_value": "windmill",
        "include": True,
        "reason": "Windmills may represent historic or culturally significant structures of visitor interest.",
    },
    {
        "category": "history_heritage",
        "osm_key": "man_made",
        "osm_value": "watermill",
        "include": True,
        "reason": "Watermills may represent historic or culturally significant structures of visitor interest.",
    },
    {
        "category": "history_heritage",
        "osm_key": "man_made",
        "osm_value": "ceremonial_gate",
        "include": True,
        "reason": "Ceremonial gates may represent culturally or historically significant landmarks.",
    },
])

feature_mapping = pd.DataFrame(FEATURE_MAPPING)
feature_mapping

,category,osm_key,osm_value,include,reason
0,attractions,tourism,attraction,True,Explicit OSM classification for tourist attrac...
1,attractions,man_made,lighthouse,True,Lighthouses represent distinctive landmarks wi...
2,attractions,man_made,observatory,True,Observatories represent distinctive visitor-or...
3,viewpoints,tourism,viewpoint,True,Explicit OSM classification for scenic viewpoi...
4,museums_culture,tourism,museum,True,Museums represent explicit visitor-oriented cu...
5,museums_culture,tourism,gallery,True,Art galleries represent cultural places potent...
6,museums_culture,tourism,artwork,True,Public artworks can represent cultural points ...
7,museums_culture,amenity,theatre,True,Theatres represent cultural and performing art...
8,history_heritage,historic,monument,True,Monuments represent explicit historical or com...
9,history_heritage,historic,memorial,True,Memorials represent commemorative places of po...


### 3.5 Nature

The `Nature` category represents natural geographic features that may provide
scenic, recreational, or exploration value to travelers.

The raw dataset inspection showed that the `natural` family is particularly
broad, with 237,602 occurrences distributed across 95 distinct values. Most
objects belong to general geographic classifications such as `tree`, `water`,
and `wood`, which are not suitable as individual recommendation candidates.

Therefore, the complete `natural=*` family is not included. Instead, the
ColMaps candidate scope retains selected feature types that may represent
meaningful places to discover during a journey.

The following observed values are initially selected:

- `natural=water`
- `natural=wetland`
- `natural=peak`
- `natural=cliff`
- `natural=bare_rock`
- `natural=beach`
- `natural=ridge`
- `natural=spring`
- `natural=cape`
- `natural=reef`
- `natural=rock`
- `natural=glacier`
- `natural=bay`
- `natural=cave_entrance`
- `natural=stone`
- `natural=desert`
- `natural=volcano`
- `natural=hot_spring`
- `natural=saddle`
- `natural=valley`
- `natural=mountain_range`
- `natural=hill`
- `natural=waterfall`
- `natural=strait`
- `natural=gulf`
- `natural=sinkhole`
- `natural=islet`
- `natural=dune`
- `natural=geyser`
- `natural=peninsula`
- `leisure=nature_reserve`
- `waterway=waterfall`
- `waterway=rapids`

In [8]:
FEATURE_MAPPING.extend([
    {
        "category": "nature",
        "osm_key": "natural",
        "osm_value": "water",
        "include": True,
        "reason": "Water bodies may represent scenic natural destinations; their specific types require subsequent validation.",
    },
    {
        "category": "nature",
        "osm_key": "natural",
        "osm_value": "wetland",
        "include": True,
        "reason": "Wetlands may represent natural areas of ecological and visitor interest.",
    },
    {
        "category": "nature",
        "osm_key": "natural",
        "osm_value": "peak",
        "include": True,
        "reason": "Mountain peaks may represent scenic or exploration-oriented destinations.",
    },
    {
        "category": "nature",
        "osm_key": "natural",
        "osm_value": "cliff",
        "include": True,
        "reason": "Cliffs may represent distinctive natural landmarks or scenic locations.",
    },
    {
        "category": "nature",
        "osm_key": "natural",
        "osm_value": "bare_rock",
        "include": True,
        "reason": "Exposed rock formations may represent distinctive natural landscapes.",
    },
    {
        "category": "nature",
        "osm_key": "natural",
        "osm_value": "beach",
        "include": True,
        "reason": "Beaches represent natural destinations potentially relevant to travelers.",
    },
    {
        "category": "nature",
        "osm_key": "natural",
        "osm_value": "ridge",
        "include": True,
        "reason": "Mountain ridges may represent scenic geographic features relevant to exploration.",
    },
    {
        "category": "nature",
        "osm_key": "natural",
        "osm_value": "spring",
        "include": True,
        "reason": "Natural springs may represent places of scenic or recreational interest.",
    },
    {
        "category": "nature",
        "osm_key": "natural",
        "osm_value": "cape",
        "include": True,
        "reason": "Capes may represent distinctive scenic coastal destinations.",
    },
    {
        "category": "nature",
        "osm_key": "natural",
        "osm_value": "reef",
        "include": True,
        "reason": "Reefs may represent distinctive natural features of visitor interest.",
    },
    {
        "category": "nature",
        "osm_key": "natural",
        "osm_value": "rock",
        "include": True,
        "reason": "Notable rock formations may represent distinctive natural landmarks.",
    },
    {
        "category": "nature",
        "osm_key": "natural",
        "osm_value": "glacier",
        "include": True,
        "reason": "Glaciers represent distinctive natural geographic features.",
    },
    {
        "category": "nature",
        "osm_key": "natural",
        "osm_value": "bay",
        "include": True,
        "reason": "Bays may represent scenic natural destinations relevant to travelers.",
    },
    {
        "category": "nature",
        "osm_key": "natural",
        "osm_value": "cave_entrance",
        "include": True,
        "reason": "Cave entrances may identify natural sites of exploration or visitor interest.",
    },
    {
        "category": "nature",
        "osm_key": "natural",
        "osm_value": "stone",
        "include": True,
        "reason": "Notable stones may represent distinctive natural landmarks.",
    },
    {
        "category": "nature",
        "osm_key": "natural",
        "osm_value": "desert",
        "include": True,
        "reason": "Desert areas may represent distinctive landscapes and exploration destinations.",
    },
    {
        "category": "nature",
        "osm_key": "natural",
        "osm_value": "volcano",
        "include": True,
        "reason": "Volcanoes represent distinctive natural landmarks and potential destinations.",
    },
    {
        "category": "nature",
        "osm_key": "natural",
        "osm_value": "hot_spring",
        "include": True,
        "reason": "Hot springs may represent natural destinations with recreational interest.",
    },
    {
        "category": "nature",
        "osm_key": "natural",
        "osm_value": "saddle",
        "include": True,
        "reason": "Mountain saddles may represent relevant geographic features for outdoor exploration.",
    },
    {
        "category": "nature",
        "osm_key": "natural",
        "osm_value": "valley",
        "include": True,
        "reason": "Valleys may represent scenic geographic destinations.",
    },
    {
        "category": "nature",
        "osm_key": "natural",
        "osm_value": "mountain_range",
        "include": True,
        "reason": "Mountain ranges represent significant natural landscapes relevant to exploration.",
    },
    {
        "category": "nature",
        "osm_key": "natural",
        "osm_value": "hill",
        "include": True,
        "reason": "Hills may represent scenic or exploration-oriented natural destinations.",
    },
    {
        "category": "nature",
        "osm_key": "natural",
        "osm_value": "waterfall",
        "include": True,
        "reason": "Waterfalls represent distinctive natural attractions suitable for route exploration.",
    },
    {
        "category": "nature",
        "osm_key": "natural",
        "osm_value": "strait",
        "include": True,
        "reason": "Straits may represent distinctive scenic geographic features.",
    },
    {
        "category": "nature",
        "osm_key": "natural",
        "osm_value": "gulf",
        "include": True,
        "reason": "Gulfs may represent significant scenic coastal features.",
    },
    {
        "category": "nature",
        "osm_key": "natural",
        "osm_value": "sinkhole",
        "include": True,
        "reason": "Sinkholes may represent distinctive geological features of exploration interest.",
    },
    {
        "category": "nature",
        "osm_key": "natural",
        "osm_value": "islet",
        "include": True,
        "reason": "Islets may represent distinctive natural destinations or scenic features.",
    },
    {
        "category": "nature",
        "osm_key": "natural",
        "osm_value": "dune",
        "include": True,
        "reason": "Dunes may represent distinctive natural landscapes of visitor interest.",
    },
    {
        "category": "nature",
        "osm_key": "natural",
        "osm_value": "geyser",
        "include": True,
        "reason": "Geysers represent distinctive natural phenomena of visitor interest.",
    },
    {
        "category": "nature",
        "osm_key": "natural",
        "osm_value": "peninsula",
        "include": True,
        "reason": "Peninsulas may represent distinctive geographic destinations with scenic value.",
    },
    {
        "category": "nature",
        "osm_key": "leisure",
        "osm_value": "nature_reserve",
        "include": True,
        "reason": "Nature reserves represent protected natural areas with potential scenic, ecological, and exploration value for travelers.",
    },
    {
        "category": "nature",
        "osm_key": "waterway",
        "osm_value": "waterfall",
        "include": True,
        "reason": "Waterfalls represent distinctive natural features with strong potential relevance for route-oriented exploration.",
    },
    {
        "category": "nature",
        "osm_key": "waterway",
        "osm_value": "rapids",
        "include": True,
        "reason": "Rapids represent distinctive natural water features potentially relevant to outdoor and nature-oriented travelers.",
    },
])

feature_mapping = pd.DataFrame(FEATURE_MAPPING)
feature_mapping

,category,osm_key,osm_value,include,reason
0,attractions,tourism,attraction,True,Explicit OSM classification for tourist attrac...
1,attractions,man_made,lighthouse,True,Lighthouses represent distinctive landmarks wi...
2,attractions,man_made,observatory,True,Observatories represent distinctive visitor-or...
3,viewpoints,tourism,viewpoint,True,Explicit OSM classification for scenic viewpoi...
4,museums_culture,tourism,museum,True,Museums represent explicit visitor-oriented cu...
5,museums_culture,tourism,gallery,True,Art galleries represent cultural places potent...
6,museums_culture,tourism,artwork,True,Public artworks can represent cultural points ...
7,museums_culture,amenity,theatre,True,Theatres represent cultural and performing art...
8,history_heritage,historic,monument,True,Monuments represent explicit historical or com...
9,history_heritage,historic,memorial,True,Memorials represent commemorative places of po...


### 3.6 Parks & Recreation

The `Parks & Recreation` category represents outdoor, recreational, and
green spaces that may provide meaningful stops during route-oriented
exploration.

The `leisure` family contains a broad range of recreational features, including
sports infrastructure, local facilities, parks, gardens, and nature-oriented
spaces. Therefore, the complete `leisure=*` family is not included.

Based on the values observed during the raw dataset inspection, the following
values are initially selected:

- `leisure=park`
- `leisure=garden`
- `leisure=picnic_table`
- `leisure=fishing`
- `leisure=resort`
- `leisure=golf_course`
- `leisure=swimming_pool`
- `leisure=sauna`
- `leisure=horse_riding`

In [9]:
FEATURE_MAPPING.extend([
    {
        "category": "parks_recreation",
        "osm_key": "leisure",
        "osm_value": "park",
        "include": True,
        "reason": "Parks represent recreational green spaces potentially relevant as stops during route exploration.",
    },
    {
        "category": "parks_recreation",
        "osm_key": "leisure",
        "osm_value": "garden",
        "include": True,
        "reason": "Gardens represent landscaped recreational spaces potentially relevant to travelers.",
    },
    {
        "category": "parks_recreation",
        "osm_key": "leisure",
        "osm_value": "picnic_table",
        "include": True,
        "reason": "Picnic tables may represent useful recreational stopping points during a journey.",
    },
    {
        "category": "parks_recreation",
        "osm_key": "leisure",
        "osm_value": "fishing",
        "include": True,
        "reason": "Fishing areas represent outdoor recreational destinations potentially relevant to traveler interests.",
    },
    {
        "category": "parks_recreation",
        "osm_key": "leisure",
        "osm_value": "resort",
        "include": True,
        "reason": "Resorts represent recreational destinations and facilities potentially relevant to travelers.",
    },
    {
        "category": "parks_recreation",
        "osm_key": "leisure",
        "osm_value": "golf_course",
        "include": True,
        "reason": "Golf courses represent recreational facilities potentially relevant to selected traveler interests.",
    },
    {
        "category": "parks_recreation",
        "osm_key": "leisure",
        "osm_value": "swimming_pool",
        "include": True,
        "reason": "Swimming pools represent recreational facilities that may be relevant to travelers and require subsequent validation.",
    },
    {
        "category": "parks_recreation",
        "osm_key": "leisure",
        "osm_value": "sauna",
        "include": True,
        "reason": "Saunas represent leisure and wellness facilities potentially relevant to traveler interests.",
    },
    {
        "category": "parks_recreation",
        "osm_key": "leisure",
        "osm_value": "horse_riding",
        "include": True,
        "reason": "Horse riding facilities represent recreational destinations potentially relevant to traveler interests.",
    }
])

feature_mapping = pd.DataFrame(FEATURE_MAPPING)
feature_mapping

,category,osm_key,osm_value,include,reason
0,attractions,tourism,attraction,True,Explicit OSM classification for tourist attrac...
1,attractions,man_made,lighthouse,True,Lighthouses represent distinctive landmarks wi...
2,attractions,man_made,observatory,True,Observatories represent distinctive visitor-or...
3,viewpoints,tourism,viewpoint,True,Explicit OSM classification for scenic viewpoi...
4,museums_culture,tourism,museum,True,Museums represent explicit visitor-oriented cu...
...,...,...,...,...,...
63,parks_recreation,leisure,resort,True,Resorts represent recreational destinations an...
64,parks_recreation,leisure,golf_course,True,Golf courses represent recreational facilities...
65,parks_recreation,leisure,swimming_pool,True,Swimming pools represent recreational faciliti...
66,parks_recreation,leisure,sauna,True,Saunas represent leisure and wellness faciliti...


### 3.7 Food & Drink

The `Food & Drink` category represents places where travelers can stop for
meals, drinks, refreshments, or similar services during a journey.

The `amenity` family contains a large and heterogeneous collection of features,
many of which are unrelated to tourism or route exploration. Therefore, only
explicit food- and drink-oriented values observed during the raw dataset
inspection are included in the candidate scope.

The following observed values are initially selected:

- `amenity=restaurant`
- `amenity=cafe`
- `amenity=fast_food`
- `amenity=bar`
- `amenity=ice_cream`
- `amenity=pub`

In [10]:
FEATURE_MAPPING.extend([
    {
        "category": "food_drink",
        "osm_key": "amenity",
        "osm_value": "restaurant",
        "include": True,
        "reason": "Restaurants represent relevant meal stops for travelers during a journey.",
    },
    {
        "category": "food_drink",
        "osm_key": "amenity",
        "osm_value": "cafe",
        "include": True,
        "reason": "Cafés represent convenient refreshment and short stopping points during route exploration.",
    },
    {
        "category": "food_drink",
        "osm_key": "amenity",
        "osm_value": "fast_food",
        "include": True,
        "reason": "Fast-food establishments provide practical meal stops during a journey.",
    },
    {
        "category": "food_drink",
        "osm_key": "amenity",
        "osm_value": "bar",
        "include": True,
        "reason": "Bars represent food and drink destinations potentially relevant to traveler interests.",
    },
    {
        "category": "food_drink",
        "osm_key": "amenity",
        "osm_value": "ice_cream",
        "include": True,
        "reason": "Ice-cream establishments may provide short refreshment stops during route exploration.",
    },
    {
        "category": "food_drink",
        "osm_key": "amenity",
        "osm_value": "pub",
        "include": True,
        "reason": "Pubs represent food and drink destinations potentially relevant to traveler interests.",
    },
])

feature_mapping = pd.DataFrame(FEATURE_MAPPING)
feature_mapping

,category,osm_key,osm_value,include,reason
0,attractions,tourism,attraction,True,Explicit OSM classification for tourist attrac...
1,attractions,man_made,lighthouse,True,Lighthouses represent distinctive landmarks wi...
2,attractions,man_made,observatory,True,Observatories represent distinctive visitor-or...
3,viewpoints,tourism,viewpoint,True,Explicit OSM classification for scenic viewpoi...
4,museums_culture,tourism,museum,True,Museums represent explicit visitor-oriented cu...
...,...,...,...,...,...
69,food_drink,amenity,cafe,True,Cafés represent convenient refreshment and sho...
70,food_drink,amenity,fast_food,True,Fast-food establishments provide practical mea...
71,food_drink,amenity,bar,True,Bars represent food and drink destinations pot...
72,food_drink,amenity,ice_cream,True,Ice-cream establishments may provide short ref...


### 3.8 Accommodation

The `Accommodation` category represents places where travelers can stay
overnight during or around their journey.

The raw dataset inspection identified several accommodation-oriented values
within the `tourism` family. Unlike the complete `tourism=*` family, these
values have a direct relationship with lodging and can therefore be grouped
under a single ColMaps user-interest category.

The following observed values are initially selected:

- `tourism=hotel`
- `tourism=hostel`
- `tourism=guest_house`
- `tourism=motel`
- `tourism=apartment`
- `tourism=chalet`
- `tourism=cabin`
- `tourism=camp_site`
- `tourism=caravan_site`
- `tourism=camp_pitch`
- `tourism=alpine_hut`
- `tourism=wilderness_hut`

In [11]:
FEATURE_MAPPING.extend([
    {
        "category": "accommodation",
        "osm_key": "tourism",
        "osm_value": "hotel",
        "include": True,
        "reason": "Hotels represent explicit traveler accommodation.",
    },
    {
        "category": "accommodation",
        "osm_key": "tourism",
        "osm_value": "hostel",
        "include": True,
        "reason": "Hostels represent traveler-oriented accommodation.",
    },
    {
        "category": "accommodation",
        "osm_key": "tourism",
        "osm_value": "guest_house",
        "include": True,
        "reason": "Guest houses represent short-term accommodation potentially relevant to travelers.",
    },
    {
        "category": "accommodation",
        "osm_key": "tourism",
        "osm_value": "motel",
        "include": True,
        "reason": "Motels provide accommodation particularly relevant to road-based travel.",
    },
    {
        "category": "accommodation",
        "osm_key": "tourism",
        "osm_value": "apartment",
        "include": True,
        "reason": "Tourism apartments represent short-term accommodation for travelers.",
    },
    {
        "category": "accommodation",
        "osm_key": "tourism",
        "osm_value": "chalet",
        "include": True,
        "reason": "Chalets represent visitor accommodation potentially relevant to recreational travel.",
    },
    {
        "category": "accommodation",
        "osm_key": "tourism",
        "osm_value": "cabin",
        "include": True,
        "reason": "Cabins represent visitor accommodation potentially relevant to nature-oriented travel.",
    },
    {
        "category": "accommodation",
        "osm_key": "tourism",
        "osm_value": "camp_site",
        "include": True,
        "reason": "Campsites provide traveler accommodation and are particularly relevant to outdoor exploration.",
    },
    {
        "category": "accommodation",
        "osm_key": "tourism",
        "osm_value": "caravan_site",
        "include": True,
        "reason": "Caravan sites provide accommodation infrastructure relevant to road-based travel.",
    },
    {
        "category": "accommodation",
        "osm_key": "tourism",
        "osm_value": "camp_pitch",
        "include": True,
        "reason": "Camp pitches represent individual camping accommodation facilities.",
    },
    {
        "category": "accommodation",
        "osm_key": "tourism",
        "osm_value": "alpine_hut",
        "include": True,
        "reason": "Alpine huts provide accommodation for travelers exploring mountainous areas.",
    },
    {
        "category": "accommodation",
        "osm_key": "tourism",
        "osm_value": "wilderness_hut",
        "include": True,
        "reason": "Wilderness huts provide basic accommodation relevant to outdoor exploration.",
    },
])

feature_mapping = pd.DataFrame(FEATURE_MAPPING)
feature_mapping

,category,osm_key,osm_value,include,reason
0,attractions,tourism,attraction,True,Explicit OSM classification for tourist attrac...
1,attractions,man_made,lighthouse,True,Lighthouses represent distinctive landmarks wi...
2,attractions,man_made,observatory,True,Observatories represent distinctive visitor-or...
3,viewpoints,tourism,viewpoint,True,Explicit OSM classification for scenic viewpoi...
4,museums_culture,tourism,museum,True,Museums represent explicit visitor-oriented cu...
...,...,...,...,...,...
81,accommodation,tourism,camp_site,True,Campsites provide traveler accommodation and a...
82,accommodation,tourism,caravan_site,True,Caravan sites provide accommodation infrastruc...
83,accommodation,tourism,camp_pitch,True,Camp pitches represent individual camping acco...
84,accommodation,tourism,alpine_hut,True,Alpine huts provide accommodation for traveler...


### 3.9 Entertainment

The `Entertainment` category represents visitor-oriented places whose primary
purpose is to provide entertainment or organized leisure experiences.

Unlike `Parks & Recreation`, which includes recreational spaces and facilities,
this category focuses on destinations that provide a specific entertainment
experience.

Based on the values observed during the raw dataset inspection, the following
values are initially selected:

- `tourism=zoo`
- `tourism=theme_park`
- `tourism=aquarium`
- `leisure=water_park`
- `amenity=nightclub`
- `amenity=casino`

In [12]:
FEATURE_MAPPING.extend([
    {
        "category": "entertainment",
        "osm_key": "tourism",
        "osm_value": "zoo",
        "include": True,
        "reason": "Zoos represent visitor-oriented entertainment and educational destinations.",
    },
    {
        "category": "entertainment",
        "osm_key": "tourism",
        "osm_value": "theme_park",
        "include": True,
        "reason": "Theme parks represent explicit visitor-oriented entertainment destinations.",
    },
    {
        "category": "entertainment",
        "osm_key": "tourism",
        "osm_value": "aquarium",
        "include": True,
        "reason": "Aquariums represent visitor-oriented entertainment and educational destinations.",
    },
    {
        "category": "entertainment",
        "osm_key": "leisure",
        "osm_value": "water_park",
        "include": True,
        "reason": "Water parks represent dedicated recreational entertainment destinations.",
    },
    {
        "category": "entertainment",
        "osm_key": "amenity",
        "osm_value": "nightclub",
        "include": True,
        "reason": "Nightclubs represent nightlife entertainment potentially relevant to traveler interests.",
    },
    {
        "category": "entertainment",
        "osm_key": "amenity",
        "osm_value": "casino",
        "include": True,
        "reason": "Casinos represent dedicated entertainment destinations potentially relevant to traveler interests.",
    },
])

feature_mapping = pd.DataFrame(FEATURE_MAPPING)
feature_mapping

,category,osm_key,osm_value,include,reason
0,attractions,tourism,attraction,True,Explicit OSM classification for tourist attrac...
1,attractions,man_made,lighthouse,True,Lighthouses represent distinctive landmarks wi...
2,attractions,man_made,observatory,True,Observatories represent distinctive visitor-or...
3,viewpoints,tourism,viewpoint,True,Explicit OSM classification for scenic viewpoi...
4,museums_culture,tourism,museum,True,Museums represent explicit visitor-oriented cu...
...,...,...,...,...,...
87,entertainment,tourism,theme_park,True,Theme parks represent explicit visitor-oriente...
88,entertainment,tourism,aquarium,True,Aquariums represent visitor-oriented entertain...
89,entertainment,leisure,water_park,True,Water parks represent dedicated recreational e...
90,entertainment,amenity,nightclub,True,Nightclubs represent nightlife entertainment p...


### 3.10 Markets & Shopping

The `Markets & Shopping` category represents commercial places that may be
useful or interesting to travelers during a journey.

Unlike tourism-oriented categories, these features primarily provide commercial
services. However, markets, shopping centers, food shops, souvenir stores, and
other selected establishments may represent useful stops or contribute to the
exploration of a destination.

The `shop` family is particularly broad and contains many everyday commercial
and specialized services. Therefore, the complete `shop=*` family is not
included.

Based on the values observed during the raw dataset inspection, the following
values are initially selected:

- `amenity=marketplace`
- `shop=convenience`
- `shop=supermarket`
- `shop=bakery`
- `shop=mall`
- `shop=greengrocer`
- `shop=alcohol`
- `shop=beverages`
- `shop=department_store`
- `shop=gift`
- `shop=confectionery`
- `shop=pastry`
- `shop=books`
- `shop=coffee`

In [13]:
FEATURE_MAPPING.extend([
    {
        "category": "markets_shopping",
        "osm_key": "amenity",
        "osm_value": "marketplace",
        "include": True,
        "reason": "Marketplaces may represent locally relevant commercial destinations and useful stops for travelers.",
    },
    {
        "category": "markets_shopping",
        "osm_key": "shop",
        "osm_value": "convenience",
        "include": True,
        "reason": "Convenience stores provide practical supplies that may be useful during a journey.",
    },
    {
        "category": "markets_shopping",
        "osm_key": "shop",
        "osm_value": "supermarket",
        "include": True,
        "reason": "Supermarkets provide practical food and travel supplies during a journey.",
    },
    {
        "category": "markets_shopping",
        "osm_key": "shop",
        "osm_value": "bakery",
        "include": True,
        "reason": "Bakeries may provide both practical food stops and locally relevant shopping experiences.",
    },
    {
        "category": "markets_shopping",
        "osm_key": "shop",
        "osm_value": "mall",
        "include": True,
        "reason": "Shopping malls represent major commercial destinations potentially useful to travelers.",
    },
    {
        "category": "markets_shopping",
        "osm_key": "shop",
        "osm_value": "greengrocer",
        "include": True,
        "reason": "Greengrocers may provide fresh local products and practical supplies for travelers.",
    },
    {
        "category": "markets_shopping",
        "osm_key": "shop",
        "osm_value": "alcohol",
        "include": True,
        "reason": "Specialized beverage shops may represent useful commercial stops for travelers.",
    },
    {
        "category": "markets_shopping",
        "osm_key": "shop",
        "osm_value": "beverages",
        "include": True,
        "reason": "Beverage shops provide refreshments and supplies potentially useful during a journey.",
    },
    {
        "category": "markets_shopping",
        "osm_key": "shop",
        "osm_value": "department_store",
        "include": True,
        "reason": "Department stores represent general shopping destinations potentially useful to travelers.",
    },
    {
        "category": "markets_shopping",
        "osm_key": "shop",
        "osm_value": "gift",
        "include": True,
        "reason": "Gift shops may provide souvenirs and locally relevant products for travelers.",
    },
    {
        "category": "markets_shopping",
        "osm_key": "shop",
        "osm_value": "confectionery",
        "include": True,
        "reason": "Confectionery shops may provide locally relevant food products and short shopping stops.",
    },
    {
        "category": "markets_shopping",
        "osm_key": "shop",
        "osm_value": "pastry",
        "include": True,
        "reason": "Pastry shops may provide local food products and convenient stops during a journey.",
    },
    {
        "category": "markets_shopping",
        "osm_key": "shop",
        "osm_value": "books",
        "include": True,
        "reason": "Bookshops may represent culturally relevant shopping destinations for selected traveler interests.",
    },
    {
        "category": "markets_shopping",
        "osm_key": "shop",
        "osm_value": "coffee",
        "include": True,
        "reason": "Coffee shops selling coffee products may represent locally relevant shopping stops for travelers.",
    },
])

feature_mapping = pd.DataFrame(FEATURE_MAPPING)
feature_mapping

,category,osm_key,osm_value,include,reason
0,attractions,tourism,attraction,True,Explicit OSM classification for tourist attrac...
1,attractions,man_made,lighthouse,True,Lighthouses represent distinctive landmarks wi...
2,attractions,man_made,observatory,True,Observatories represent distinctive visitor-or...
3,viewpoints,tourism,viewpoint,True,Explicit OSM classification for scenic viewpoi...
4,museums_culture,tourism,museum,True,Museums represent explicit visitor-oriented cu...
...,...,...,...,...,...
101,markets_shopping,shop,gift,True,Gift shops may provide souvenirs and locally r...
102,markets_shopping,shop,confectionery,True,Confectionery shops may provide locally releva...
103,markets_shopping,shop,pastry,True,Pastry shops may provide local food products a...
104,markets_shopping,shop,books,True,Bookshops may represent culturally relevant sh...


### 3.11 Health & Pharmacy

The `Health & Pharmacy` category represents healthcare services that may be
useful to travelers during a journey.

Unlike tourism-oriented categories, these features are primarily functional
services. However, access to pharmacies, hospitals, clinics, and medical
professionals may be important when traveling between locations, especially
along longer routes or through unfamiliar areas.

The broader `amenity` family contains many unrelated services, so only explicit
healthcare-oriented classifications are included.

Based on the values observed during the raw dataset inspection, the following
values are initially selected:

- `amenity=pharmacy`
- `amenity=hospital`
- `amenity=clinic`
- `amenity=doctors`

In [14]:
FEATURE_MAPPING.extend([
    {
        "category": "health_pharmacy",
        "osm_key": "amenity",
        "osm_value": "pharmacy",
        "include": True,
        "reason": "Pharmacies provide essential medication and health-related services that may be useful during a journey.",
    },
    {
        "category": "health_pharmacy",
        "osm_key": "amenity",
        "osm_value": "hospital",
        "include": True,
        "reason": "Hospitals provide essential healthcare services that may be important to travelers.",
    },
    {
        "category": "health_pharmacy",
        "osm_key": "amenity",
        "osm_value": "clinic",
        "include": True,
        "reason": "Clinics provide healthcare services that may be useful during travel.",
    },
    {
        "category": "health_pharmacy",
        "osm_key": "amenity",
        "osm_value": "doctors",
        "include": True,
        "reason": "Medical practices provide healthcare access that may be useful to travelers.",
    },
])

feature_mapping = pd.DataFrame(FEATURE_MAPPING)
feature_mapping

,category,osm_key,osm_value,include,reason
0,attractions,tourism,attraction,True,Explicit OSM classification for tourist attrac...
1,attractions,man_made,lighthouse,True,Lighthouses represent distinctive landmarks wi...
2,attractions,man_made,observatory,True,Observatories represent distinctive visitor-or...
3,viewpoints,tourism,viewpoint,True,Explicit OSM classification for scenic viewpoi...
4,museums_culture,tourism,museum,True,Museums represent explicit visitor-oriented cu...
...,...,...,...,...,...
105,markets_shopping,shop,coffee,True,Coffee shops selling coffee products may repre...
106,health_pharmacy,amenity,pharmacy,True,Pharmacies provide essential medication and he...
107,health_pharmacy,amenity,hospital,True,Hospitals provide essential healthcare service...
108,health_pharmacy,amenity,clinic,True,Clinics provide healthcare services that may b...


### 3.12 Transport & Travel

The `Transport & Travel` category represents transport-related facilities and
services that may be useful while traveling between an origin and destination.

Unlike the discovery-oriented categories, these features primarily support the
journey itself. They may help users identify locations for refueling, parking,
accessing public transport, using ferry connections, or other practical travel
needs.

OpenStreetMap contains extensive transport and road infrastructure that is not
directly relevant to the ColMaps recommendation model. Therefore, the category
does not attempt to represent the complete transportation network. Only
selected facilities that may constitute useful stops or destinations during a
journey are included.

Based on the values observed during the raw dataset inspection, the following
values are initially selected:

- `amenity=fuel`
- `amenity=parking`
- `amenity=parking_entrance`
- `amenity=bus_station`
- `amenity=car_wash`

In [15]:
FEATURE_MAPPING.extend([
    {
        "category": "transport_travel",
        "osm_key": "amenity",
        "osm_value": "fuel",
        "include": True,
        "reason": "Fuel stations provide an important practical service for travelers using road-based routes.",
    },
    {
        "category": "transport_travel",
        "osm_key": "amenity",
        "osm_value": "parking",
        "include": True,
        "reason": "Parking facilities may be useful when stopping at destinations or points of interest along a route.",
    },
    {
        "category": "transport_travel",
        "osm_key": "amenity",
        "osm_value": "parking_entrance",
        "include": True,
        "reason": "Parking entrances may provide useful access information for parking facilities near route destinations.",
    },
    {
        "category": "transport_travel",
        "osm_key": "amenity",
        "osm_value": "bus_station",
        "include": True,
        "reason": "Bus stations represent transport facilities potentially useful to travelers during a journey.",
    },
    {
        "category": "transport_travel",
        "osm_key": "amenity",
        "osm_value": "car_wash",
        "include": True,
        "reason": "Car washes represent optional vehicle-related services that may be useful during road travel.",
    },
])

feature_mapping = pd.DataFrame(FEATURE_MAPPING)
feature_mapping

,category,osm_key,osm_value,include,reason
0,attractions,tourism,attraction,True,Explicit OSM classification for tourist attrac...
1,attractions,man_made,lighthouse,True,Lighthouses represent distinctive landmarks wi...
2,attractions,man_made,observatory,True,Observatories represent distinctive visitor-or...
3,viewpoints,tourism,viewpoint,True,Explicit OSM classification for scenic viewpoi...
4,museums_culture,tourism,museum,True,Museums represent explicit visitor-oriented cu...
...,...,...,...,...,...
110,transport_travel,amenity,fuel,True,Fuel stations provide an important practical s...
111,transport_travel,amenity,parking,True,Parking facilities may be useful when stopping...
112,transport_travel,amenity,parking_entrance,True,Parking entrances may provide useful access in...
113,transport_travel,amenity,bus_station,True,Bus stations represent transport facilities po...


## 4. Feature Scope Summary

Before proceeding to the validation of individual OSM features, the resulting
ColMaps feature scope is summarized and checked for basic structural
consistency.

At this stage, the objective is not to evaluate the quality or suitability of
individual OSM objects. Instead, the following checks provide a concise overview
of the semantic model defined in this notebook and verify that the mapping does
not contain accidental duplicates or inconsistencies between the declared
ColMaps categories and their corresponding OSM mappings.

The current scope can therefore be considered a candidate feature model. Its
individual mappings will be examined against the actual OSM objects in the
subsequent validation stage.


In [16]:
## Quick sample of tag distribution 
print(f"Total selected mappings: {len(feature_mapping)}")
print(f"ColMaps categories: {feature_mapping['category'].nunique()}")
print(f"OSM keys used: {feature_mapping['osm_key'].nunique()}")

feature_mapping.groupby("category").size().sort_values(ascending=False)

Total selected mappings: 115
ColMaps categories: 12
OSM keys used: 8


category
nature              33
history_heritage    18
markets_shopping    14
accommodation       12
parks_recreation     9
entertainment        6
food_drink           6
transport_travel     5
museums_culture      4
health_pharmacy      4
attractions          3
viewpoints           1
dtype: int64

In [17]:
duplicates = feature_mapping[
    feature_mapping.duplicated(
        subset=["category", "osm_key", "osm_value"],
        keep=False,
    )
]

print(f"Duplicate mappings: {len(duplicates)}")

duplicates

Duplicate mappings: 0


,category,osm_key,osm_value,include,reason


In [18]:
declared_categories = set(FEATURE_CATEGORIES.keys())
mapped_categories = set(feature_mapping["category"].unique())

print("Declared categories:", len(declared_categories))
print("Mapped categories:", len(mapped_categories))
print("Categories without mappings:", declared_categories - mapped_categories)
print("Unknown mapped categories:", mapped_categories - declared_categories)

Declared categories: 12
Mapped categories: 12
Categories without mappings: set()
Unknown mapped categories: set()


## 5. Conclusion

This notebook established the initial semantic feature scope of ColMaps by
mapping application-level user interests to explicit OpenStreetMap
classifications.

The resulting candidate model contains **115 OSM `key=value` mappings**
distributed across **12 ColMaps categories** and originating from **8 OSM tag
families**. Structural consistency checks confirmed that all declared
categories are represented in the mapping and that no duplicate mappings are
present.

The resulting scope intentionally includes both discovery-oriented places and
practical traveler services. This allows ColMaps to represent not only
attractions, cultural places, natural features, and recreational destinations,
but also services such as accommodation, food, shopping, healthcare, and
transport-related facilities that may be useful during a journey.

However, the mappings defined in this notebook represent a **candidate feature
scope rather than the final dataset specification**. Inclusion at this stage
indicates that an OSM classification is semantically relevant to the ColMaps
use case; it does not imply that every OSM object belonging to that
classification is suitable for inclusion in the application.

The next stage will therefore examine the selected feature types against the
actual OSM data in greater detail. This validation will consider characteristics
such as available attributes, naming, geometry, secondary tags, object
distribution, potential duplicates, and the practical usefulness of individual
classifications for route-oriented recommendations.

As a result of this validation, some mappings may be refined or excluded, and
broad classifications may require additional filtering rules. Consequently, the
final number of retained mappings and the volume of geographic objects imported
into the ColMaps dataset may be lower than the candidate scope defined here.

This separation between **semantic scope definition** and **data-driven
validation** ensures that filtering decisions are based on observed dataset
characteristics rather than assumptions made before examining the selected OSM
features.
